# Notebook 1: QSARmil multi-conformer consensus modelling

This notebook introduces the high-level API for building multi-conformer models with **QSARmil**. The API is designed to simplify benchmarking QSARmil against alternative approaches without requiring detailed knowledge of the underlying modelling configuration. As a result, the workflow remains concise and easy to follow. You can adapt the code below to your own dataset and run the complete **QSARmil** modelling pipeline with minimal modifications.

In [1]:
import pandas as pd
from sklearn.metrics import r2_score
from qsarmil.meta import MultiConformerRegressor, MultiConformerClassifier 

### 1. Load your trainign data

As an example, we use a publicly available and easily accessible collection of molecular bioactivity datasets introduced in the paper by:

> Van Tilborg, Derek, Alisa Alenicheva, and Francesca Grisoni. "Exposing the limitations of molecular machine learning with activity cliffs." Journal of chemical information and modeling 62.23 (2022): 5938-5951.

From this collection, we select one dataset for demonstration purposes.

In [2]:
# load data
url = "https://raw.githubusercontent.com/molML/MoleculeACE/main/MoleculeACE/Data/benchmark_data/CHEMBL2034_Ki.csv"
df_ace = pd.read_csv(url)

# train/test split
df_train = df_ace[df_ace["split"] == "train"][["smiles", "y"]]
df_test = df_ace[df_ace["split"] == "test"][["smiles", "y"]]

In [3]:
# uncomment for quick testing
df_train = df_train.sample(frac=0.1, random_state=42).reset_index(drop=True)
df_test = df_test.sample(frac=0.1, random_state=42).reset_index(drop=True)
df_train.shape, df_test.shape

((60, 2), (15, 2))

### 2. Build multi-conformer consensus model

The multi-conformer model building pipeline consists of several sequential steps:

 - Conformer generation: the maximum number of conformers is defined using the ``num_conf`` parameter.
 - Descriptor calculation: by default, multiple types of 3D molecular descriptors are computed.
 - Model training: several multi-instance learning networks are used to train models. Internal stepwise hyperparameter optimization can be enabled with ``hopt=True`` (note that this increases runtime).
 - Consensus search: once multiple multi-conformer models are trained, a genetic algorithm is applied to identify an optimal consensus model.

Task type is defined automatically (regression or binary classification), ``output_folder`` (directory for storing predictions from individual models), and ``verbose`` (controls the level of pipeline progress output).

In [4]:
smiles_train, y_train = df_train["smiles"].to_list(), df_train["y"].to_list()

In [5]:
model = MultiConformerRegressor(num_conf=10, hopt=False, verbose=True, output_folder="./mcfm")
model.train(smiles_train, y_train)
model.save()


++++++++++++++++++++++++++
Step-1. SMILES parsing
++++++++++++++++++++++++++
> For 60 of 60 molecules, SMILES were parsed correctly.

++++++++++++++++++++++++++
Step-2. Conformer generation
++++++++++++++++++++++++++
> For 60 of 60 molecules, conformers were generated successfully.
> Average num conf: 10.0 | min num conf: 10 | max num conf: 10

++++++++++++++++++++++++++
Step-3. Descriptor calculation
++++++++++++++++++++++++++
[1/9] RDKitGEOM:
      > Finished in 0.00 min | Memory usage: 0.296 G
[2/9] RDKitAUTOCORR:
      > Finished in 0.01 min | Memory usage: 0.341 G
[3/9] RDKitRDF:
      > Finished in 0.01 min | Memory usage: 0.345 G
[4/9] RDKitMORSE:
      > Finished in 0.02 min | Memory usage: 0.346 G
[5/9] RDKitWHIM:
      > Finished in 0.00 min | Memory usage: 0.346 G
[6/9] MolFeatUSRD:
      > Finished in 0.00 min | Memory usage: 0.547 G
[7/9] MolFeatElectroShape:
      > Finished in 0.01 min | Memory usage: 0.560 G
[8/9] RDKitGETAWAY:
      > Finished in 0.06 min | Memory usa

### 3. Predict target property for new molecules

In [6]:
smiles_test, y_test = df_test["smiles"].to_list(), df_test["y"].to_list()

In [8]:
model = MultiConformerRegressor.load("./mcfm/model.pkl")
y_pred = model.predict(smiles_test)

In [10]:
r2_score(y_test, y_pred)

0.028506427296122072